# Style Matcher Golden Model (YCbCr-based)

This notebook implements a **hardware-friendly style matcher** pipeline:

1. Gray-world white balance (WB)
2. Global color statistics 
3. YCbCr-based color transfer:
   - Full 256-bin histogram matching on Y (luma)
   - Mean/std matching on Cb/Cr (chroma)

The design avoids heavy libraries (no MKL, no ML), so it can be mapped to RTL:
- Only additions, multiplications, divisions (for gain) and LUT lookups.
- All operations are suitable for fixed-point hardware implementation.


## Reference

In [ ]:
ref_idx = 0

In [11]:
import numpy as np

quan_bit = 8

## DNG to RGB

In [12]:
import rawpy
import imageio.v2 as imageio
import numpy as np
import cv2

def quan(num: np.ndarray, bit = 10):
    return np.int64(num * 2**bit) / 2**bit

def dec_to_hex(num, bit=16):
    num = hex(int(num))
    num_str = str(num)[2:]
    while (len(num_str) < bit // 4):
        num_str = "0" + num_str
    return num_str

def int_exp(num, bit=2):
    num = str(num)
    while (len(num) < bit):
        num = "0" + num
    return num

def demosaic_rggb_bilinear(bayer: np.ndarray) -> np.ndarray:
    """bayer: H x W, RGGB, uint16/float32 -> 回傳 H x W x 3, float32"""
    H, W = bayer.shape
    pad = np.pad(bayer.astype(np.float32), 1, mode='edge')

    R = np.zeros((H, W), dtype=np.float32)
    G = np.zeros_like(R)
    B = np.zeros_like(R)

    for y in range(H):
        for x in range(W):
            i, j = y + 1, x + 1  # pad 之後的座標

            if (y % 2 == 0) and (x % 2 == 0):
                # R 位置
                R[y, x] = pad[i, j]
                G[y, x] = (pad[i-1, j] + pad[i+1, j] + pad[i, j-1] + pad[i, j+1]) / 4.0
                B[y, x] = (pad[i-1, j-1] + pad[i-1, j+1] + pad[i+1, j-1] + pad[i+1, j+1]) / 4.0

            elif (y % 2 == 1) and (x % 2 == 1):
                # B 位置
                B[y, x] = pad[i, j]
                G[y, x] = (pad[i-1, j] + pad[i+1, j] + pad[i, j-1] + pad[i, j+1]) / 4.0
                R[y, x] = (pad[i-1, j-1] + pad[i-1, j+1] + pad[i+1, j-1] + pad[i+1, j+1]) / 4.0

            elif (y % 2 == 0) and (x % 2 == 1):
                # G，位在 R row
                G[y, x] = pad[i, j]
                R[y, x] = (pad[i, j-1] + pad[i, j+1]) / 2.0
                B[y, x] = (pad[i-1, j] + pad[i+1, j]) / 2.0

            else:
                # (y % 2 == 1) and (x % 2 == 0)
                # G，位在 B row
                G[y, x] = pad[i, j]
                R[y, x] = (pad[i-1, j] + pad[i+1, j]) / 2.0
                B[y, x] = (pad[i, j-1] + pad[i, j+1]) / 2.0

    rgb = np.stack([R, G, B], axis=-1)  # H x W x 3
    return rgb

# 假設前面你已經得到 rgb_lin (H, W, 3), float32，單位仍然是 linear RAW 值
# 例如從 demosaic_rggb_bilinear() 出來的結果

def apply_better_gamma(rgb_lin, white_percent=99.5, gamma=2.2):
    """
    rgb_lin : H x W x 3, float32, 線性 RGB
    raw     : rawpy 的物件，用來拿 black level / saturation 等
    """

    rgb = rgb_lin.copy()

    # 1) 扣除黑階（用平均 black level）
    # raw.black_level_per_channel 通常是 4 個值（對應 RGBG 四個）
    # black = float(np.mean(raw.black_level_per_channel))
    # print(black)
    # rgb = rgb - black
    # rgb[rgb < 0] = 0.0

    # 2) 自動曝光：用 99.5 百分位數當白點，而不是用 max
    #   這樣被燈管/高光 clip 掉的一小部分不會影響整張圖
    #   避免 0 把百分位數拉下來，用 rgb>0 的區域算
    flat = rgb.reshape(-1, 3)
    mag = flat.max(axis=1)             # 各像素取 max(R,G,B) 當亮度
    mag = mag[mag > 0]

    if mag.size == 0:
        white_point = 1.0
    else:
        bin = 256
        step = 20
        cnt = 0
        m = 0
        hist = [0] * bin
        while m < len(mag):
            for i in range(bin):
                if (i*(2**14/bin) <= mag[m] and mag[m] < (i+1)*(2**14/bin)):
                    hist[i] += 1
                    break
            if cnt == 1920 // step - 1:
                m += 1920 * (step - 1)
                cnt = 0
            else:
                cnt += 1
            m += step
        cdf = np.cumsum(hist).astype(np.float64)
        white_point1 = np.searchsorted(cdf, int(1920*1080/(step**2)*white_percent/100), side="left")*2**14/bin
        # white_point2 = np.percentile(mag, white_percent)
        white_point = white_point1

    if white_point <= 0:
        white_point = 1.0

    white_point_inv = quan(1/white_point, 22)
    rgb = rgb * white_point_inv
    rgb = np.clip(rgb, 0.0, 1.0)
    rgb = quan(rgb)

    # 3) sRGB gamma
    #    可以用純 power-law x^(1/gamma)，也可以用正式 sRGB curve
    #    下面給你正式 sRGB 版
    a = 0.0546875 # 0.0000111
    # threshold = 0.0031308

    # srgb = np.where(
    #     1 == 2,
    #     12.92 * rgb,
    #     (1 + a) * np.power(rgb, 1.0 / 2.4) - a
    # )
    srgb = quan((1 + a) * np.power(rgb, 1.4 / 2.4) - a)

    srgb = np.clip(srgb, 0.0, 1.0)

    srgb *= 255.0 + 0.5

    # 4) 轉成 8-bit
    rgb8 = (srgb).astype(np.uint8)
    return rgb8


# file = "251204_153240"
# in_path  = f"C:/Users/0607f/code/python/ICLAB/Final/{file}_CW_14mm/{file}_CW_14mm_000000.dng"


## RGB ↔ YCbCr Conversion (BT.601 full range, BGR order as in OpenCV)

In [13]:
def rgb_to_ycbcr_bgr(img_bgr: np.ndarray) -> np.ndarray:
    """Convert BGR (as from OpenCV) to YCbCr (BT.601 full range).

    Input : HxWx3 uint8 (BGR)
    Output: HxWx3 float32 (Y, Cb, Cr) in [0,255]
    """
    assert img_bgr.ndim == 3 and img_bgr.shape[2] == 3
    img = img_bgr.astype(np.float32)
    R = img[..., 0]
    G = img[..., 1]
    B = img[..., 2]

    Y  = quan(0.299)  * R + quan(0.587)  * G + quan(0.114)  * B
    Cb = quan(-0.1687) * R - quan(0.3313) * G + quan(0.5)    * B + 128.0
    Cr = quan(0.5)    * R - quan(0.4187) * G - quan(0.0813) * B + 128.0

    Y = quan(Y, 0)
    Cb = quan(Cb, 8)
    Cr = quan(Cr, 8)

    Y  = np.clip(Y,  0, 255)        # Y to uint8
    Cb = np.clip(Cb, 0, 255)
    Cr = np.clip(Cr, 0, 255)

    return np.stack([Y, Cb, Cr], axis=-1).astype(np.float32)


def ycbcr_to_rgb_bgr(img_ycbcr: np.ndarray) -> np.ndarray:
    """Convert YCbCr (BT.601 full range) back to BGR (for OpenCV).

    Input : HxWx3 float32 (Y, Cb, Cr)
    Output: HxWx3 uint8 (BGR)
    """
    assert img_ycbcr.ndim == 3 and img_ycbcr.shape[2] == 3
    Y  = img_ycbcr[..., 0]
    Cb = img_ycbcr[..., 1] - 128.0
    Cr = img_ycbcr[..., 2] - 128.0

    R = Y + quan(1.402)   * Cr
    G = Y - quan(0.34414) * Cb - quan(0.71414) * Cr
    B = Y + quan(1.772)   * Cb

    R = np.clip(R, 0, 255)
    G = np.clip(G, 0, 255)
    B = np.clip(B, 0, 255)

    return np.stack([B, G, R], axis=-1).astype(np.uint8)


## Step 4 (part 1) – 256-bin Histogram Matching on Y (Luma)

In [14]:
def compute_y_lut_from_reference(Y_src: np.ndarray, Y_ref: np.ndarray) -> np.ndarray:
    """Compute a 256-entry LUT for histogram matching of Y channel.

    Uses full 256-bin histograms and CDF matching.
    """
    assert Y_src.dtype == np.uint8 and Y_ref.dtype == np.uint8

    src_flat = Y_src.flatten()
    ref_flat = Y_ref.flatten()


    bin = 256
    hist_ref = [0] * bin
    hist_src = [0] * bin
    step = 20
    cnt = 0
    m = 0

    while m < len(src_flat):
        hist_src[src_flat[m]] += 1
        hist_ref[ref_flat[m]] += 1

        if cnt == 1920 // step - 1:
            m += 1920 * (step - 1)
            cnt = 0
        else:
            cnt += 1
        m += step

    # hist_src_t, _ = np.histogram(src_flat, bins=256, range=(0, 256))
    # hist_ref_t, _ = np.histogram(ref_flat, bins=256, range=(0, 256))

    cdf_src = np.cumsum(hist_src).astype(np.uint16)
    cdf_ref = np.cumsum(hist_ref).astype(np.uint16)

    # cdf_src_t = np.cumsum(hist_src_t).astype(np.float64)
    # cdf_ref_t = np.cumsum(hist_ref_t).astype(np.float64)

    # if cdf_src[-1] > 0:
    #     cdf_src /= cdf_src[-1]
    # if cdf_ref[-1] > 0:
    #     cdf_ref /= cdf_ref[-1]

    # if cdf_src_t[-1] > 0:
    #     cdf_src_t /= cdf_src_t[-1]
    # if cdf_ref_t[-1] > 0:
    #     cdf_ref_t /= cdf_ref_t[-1]

    LUT = np.zeros(256, dtype=np.uint8)
    for g in range(256):
        v = cdf_src[g]
        h = np.searchsorted(cdf_ref, v, side="left")
        if h > 255:
            h = 255
        LUT[g] = h

    return LUT
    

def apply_y_lut(Y: np.ndarray, LUT: np.ndarray) -> np.ndarray:
    """Apply a 256-entry LUT to Y channel (uint8)."""
    assert Y.dtype == np.uint8 and LUT.shape == (256,)
    flat = Y.flatten()
    out_flat = LUT[flat]
    return out_flat.reshape(Y.shape)


## Step 4 (part 2) – Mean/Std Matching on Cb/Cr (Chroma)

In [15]:
def compute_chroma_stats(ycbcr: np.ndarray):
    """Compute mean and std of Cb and Cr from YCbCr image.

    Returns (mu_Cb, sigma_Cb, mu_Cr, sigma_Cr).
    """
    assert ycbcr.ndim == 3 and ycbcr.shape[2] == 3
    Cb = ycbcr[..., 1].astype(np.float32)
    Cr = ycbcr[..., 2].astype(np.float32)

    # Cb *= 2**8
    # Cr *= 2**8

    # Cb = np.uint32(Cb)
    # Cr = np.uint32(Cr)

    Cb = np.uint64(Cb * 2**8)
    Cr = np.uint64(Cr * 2**8)
    mu_Cb = quan(Cb.mean() / 2**8, 10)
    mu_Cr = quan(Cr.mean() / 2**8, 10)
    Cb_sqrt = Cb**2
    Cr_sqrt = Cr**2
    mu_Cb_sqrt = quan(Cb_sqrt.mean() / 2**16, 10)
    mu_Cr_sqrt = quan(Cr_sqrt.mean() / 2**16, 10)
    # mu_Cb_sqrt = quan(Cb_sqrt.sum()/2025 / 2**26, 10)
    # mu_Cr_sqrt = quan(Cr_sqrt.sum()/2025 / 2**26, 10)
    sigma_Cb = quan((mu_Cb_sqrt - quan(mu_Cb**2, 10)), 10)
    sigma_Cr = quan((mu_Cr_sqrt - quan(mu_Cr**2, 10)), 10)

    return mu_Cb, sigma_Cb, mu_Cr, sigma_Cr


def apply_chroma_mean_std_match(ycbcr_src: np.ndarray,
                                stats_src: tuple,
                                stats_ref: tuple) -> np.ndarray:
    """Apply 1D Gaussian mean/std matching on Cb/Cr.

    For each channel:
      Cb' = (Cb - mu_src) * (sigma_ref / (sigma_src+eps)) + mu_ref
      Cr' = (Cr - mu_src) * (sigma_ref / (sigma_src+eps)) + mu_ref
    """
    assert ycbcr_src.ndim == 3 and ycbcr_src.shape[2] == 3
    Y  = ycbcr_src[..., 0]
    Cb = quan(ycbcr_src[..., 1].astype(np.float32), quan_bit)
    Cr = quan(ycbcr_src[..., 2].astype(np.float32), quan_bit)

    mu_Cb_s, sigma_Cb_s, mu_Cr_s, sigma_Cr_s = stats_src
    mu_Cb_r, sigma_Cb_r, mu_Cr_r, sigma_Cr_r = stats_ref

    # eps = 2**-quan_bit
    g_Cb = quan(quan(sigma_Cb_r / sigma_Cb_s, 6)**0.5, 10) if sigma_Cb_s > 0 else 1.0
    g_Cr = quan(quan(sigma_Cr_r / sigma_Cr_s, 6)**0.5, 10) if sigma_Cr_s > 0 else 1.0

    Cb_out = quan((Cb - mu_Cb_s) * g_Cb + mu_Cb_r, quan_bit)
    Cr_out = quan((Cr - mu_Cr_s) * g_Cr + mu_Cr_r, quan_bit)  

    Cb_out = np.clip(Cb_out, 0, 255)
    Cr_out = np.clip(Cr_out, 0, 255)

    out = np.stack([Y, Cb_out, Cr_out], axis=-1).astype(np.float32)
    return out



## Advanced Step 4 – YCbCr-based Color Transfer

In [16]:
def color_transfer_ycbcr(src_bgr: np.ndarray, ref_bgr: np.ndarray) -> np.ndarray:
    """Perform advanced color transfer in YCbCr:
    - Convert src/ref to YCbCr.
    - 256-bin histogram matching on Y.
    - Mean/std matching on Cb/Cr.
    - Convert back to BGR.
    """
    src_ycbcr = rgb_to_ycbcr_bgr(src_bgr)
    ref_ycbcr = rgb_to_ycbcr_bgr(ref_bgr)

    Y_src = src_ycbcr[..., 0].astype(np.uint8)
    Y_ref = ref_ycbcr[..., 0].astype(np.uint8)

    LUT_Y = compute_y_lut_from_reference(Y_src, Y_ref)
    Y_matched = apply_y_lut(Y_src, LUT_Y).astype(np.float32)

    src_ycbcr_matched = src_ycbcr.copy()
    src_ycbcr_matched[..., 0] = Y_matched

    stats_src = compute_chroma_stats(src_ycbcr_matched)
    stats_ref = compute_chroma_stats(ref_ycbcr)

    src_ycbcr_chroma_matched = apply_chroma_mean_std_match(src_ycbcr_matched, stats_src, stats_ref)

    out_bgr = ycbcr_to_rgb_bgr(src_ycbcr_chroma_matched)
    return out_bgr


## Full Color Matcher Pipeline (Steps 2 → 3 → 4)

In [17]:
def color_match_pipeline_debug(
    src_bgr,
    ref_bgr,
    use_wb=False,
    use_ccm=False,  # 你現在不用 CCM
    ccm_matrix=None,
    ccm_bias=None,
    H = 1080,
    W = 1920,
    _ = 0
):
    results = {}  # 存各階段影像
    
    # ======= Step 1: RGB → YCbCr =======
    ycbcr = rgb_to_ycbcr_bgr(src_bgr)
    # 為了能看：轉回 RGB
    ycbcr_rgb_view = ycbcr_to_rgb_bgr(ycbcr)
    results["ycbcr"] = ycbcr_rgb_view.copy()

    # ======= Step 2: Y Histogram Matching =======
    src_y = ycbcr[..., 0].astype(np.uint8)
    ref_y = rgb_to_ycbcr_bgr(ref_bgr)[..., 0].astype(np.uint8)
    ref_cbcr = rgb_to_ycbcr_bgr(ref_bgr)[..., 1:3].astype(np.float32)

    LUT_Y = compute_y_lut_from_reference(src_y, ref_y)
    y_matched = apply_y_lut(src_y, LUT_Y).astype(np.float32)

    ycbcr_step3 = ycbcr.copy()
    ycbcr_step3[..., 0] = y_matched
    results["after_hist"] = ycbcr_to_rgb_bgr(ycbcr_step3).copy()

    # ======= Step 3: Cb/Cr mean-std matching =======
    stats_src = compute_chroma_stats(ycbcr_step3)
    stats_ref = compute_chroma_stats(rgb_to_ycbcr_bgr(ref_bgr))
    ycbcr_final = apply_chroma_mean_std_match(ycbcr_step3, stats_src, stats_ref)

    final_bgr = ycbcr_to_rgb_bgr(ycbcr_final)
    results["final"] = final_bgr.copy()

    # ======= Write Golden =======

    file_str_r_ref = ["", "", "", ""]
    file_str_gb_ref = ["", "", "", ""]
    file_str_y_src = ["", "", "", ""]
    file_str_cbcr_src = ["", "", "", ""]
    file_str_y_ref = ["", "", "", ""]
    file_str_cbcr_ref = ["", "", "", ""]
    file_str_y_lumi = ["", "", "", ""]
    file_str_cbcr_color = ["", "", "", ""]
    file_str_y_final = ["", "", "", ""]
    file_str_cbcr_final = ["", "", "", ""]

    # for i in range(H):
    #     for j in range(W // 4):
    #         if _ == 0:
    #             file_str_r_ref[i % 4] += "".join(map(lambda x: dec_to_hex(x, 8), ref_bgr[:,:,0][i][j*4:j*4+4])) + "\n"
    #             file_str_gb_ref[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(ref_bgr[:,:,1][i][j*4])*2**16 + int(ref_bgr[:,:,2][i][j*4]), 
    #                                                                                     int(ref_bgr[:,:,1][i][j*4+1])*2**16 + int(ref_bgr[:,:,2][i][j*4+1]),
    #                                                                                     int(ref_bgr[:,:,1][i][j*4+2])*2**16 + int(ref_bgr[:,:,2][i][j*4+2]),
    #                                                                                     int(ref_bgr[:,:,1][i][j*4+3])*2**16 + int(ref_bgr[:,:,2][i][j*4+3])])) + "\n"
    #         file_str_y_src[i % 4] += "".join(map(dec_to_hex, ycbcr[:,:,0][i][j*4:j*4+4])) + "\n"
    #         file_str_cbcr_src[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(ycbcr[:,:,1][i][j*4]*2**24) + int(ycbcr[:,:,2][i][j*4]*2**8), 
    #                                                                                 int(ycbcr[:,:,1][i][j*4+1]*2**24) + int(ycbcr[:,:,2][i][j*4+1]*2**8),
    #                                                                                 int(ycbcr[:,:,1][i][j*4+2]*2**24) + int(ycbcr[:,:,2][i][j*4+2]*2**8),
    #                                                                                 int(ycbcr[:,:,1][i][j*4+3]*2**24) + int(ycbcr[:,:,2][i][j*4+3]*2**8)])) + "\n"
    #         file_str_y_ref[i % 4] += "".join(map(lambda x: dec_to_hex(x, 8), ref_y[i][j*4:j*4+4])) + "\n"
    #         file_str_cbcr_ref[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(ref_cbcr[:,:,0][i][j*4]*2**24) + int(ref_cbcr[:,:,1][i][j*4]*2**8), 
    #                                                                                 int(ref_cbcr[:,:,0][i][j*4+1]*2**24) + int(ref_cbcr[:,:,1][i][j*4+1]*2**8),
    #                                                                                 int(ref_cbcr[:,:,0][i][j*4+2]*2**24) + int(ref_cbcr[:,:,1][i][j*4+2]*2**8),
    #                                                                                 int(ref_cbcr[:,:,0][i][j*4+3]*2**24) + int(ref_cbcr[:,:,1][i][j*4+3]*2**8)])) + "\n"
    #         file_str_y_lumi[i % 4] += "".join(map(dec_to_hex, y_matched[i][j*4:j*4+4])) + "\n"
    #         file_str_cbcr_color[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(ycbcr_final[:,:,1][i][j*4]*2**24)   + int(ycbcr_final[:,:,2][i][j*4]*2**8), 
    #                                                                                 int(ycbcr_final[:,:,1][i][j*4+1]*2**24) + int(ycbcr_final[:,:,2][i][j*4+1]*2**8),
    #                                                                                 int(ycbcr_final[:,:,1][i][j*4+2]*2**24) + int(ycbcr_final[:,:,2][i][j*4+2]*2**8),
    #                                                                                 int(ycbcr_final[:,:,1][i][j*4+3]*2**24) + int(ycbcr_final[:,:,2][i][j*4+3]*2**8)])) + "\n"
    #         file_str_y_final[i % 4] += "".join(map(dec_to_hex, final_bgr[:,:,2][i][j*4:j*4+4])) + "\n"
    #         file_str_cbcr_final[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(final_bgr[:,:,1][i][j*4])*2**16   + int(final_bgr[:,:,0][i][j*4]), 
    #                                                                                 int(final_bgr[:,:,1][i][j*4+1])*2**16 + int(final_bgr[:,:,0][i][j*4+1]),
    #                                                                                 int(final_bgr[:,:,1][i][j*4+2])*2**16 + int(final_bgr[:,:,0][i][j*4+2]),
    #                                                                                 int(final_bgr[:,:,1][i][j*4+3])*2**16 + int(final_bgr[:,:,0][i][j*4+3])])) + "\n"
            
    #     print (f"{_}th: {int(round(i/H, 2) * 100)}%")
    # for b in range(4):
    #     if _ == 0:
    #         with open(f"pat/ref_pat_0{ref_idx}_r{b}.dat", "w") as f:
    #             f.write(file_str_r_ref[b])
    #         with open(f"pat/ref_pat_0{ref_idx}_gb{b}.dat", "w") as f:
    #             f.write(file_str_gb_ref[b])            
    #     with open(f"golden/ref_y_channel/ref_ych_{int_exp(_)}_fy{b}.dat", "w") as f:
    #         f.write(file_str_y_ref[b])
    #     with open(f"golden/ref_cbcr_channel/ref_cbcrch_{int_exp(_)}_fc{b}.dat", "w") as f:
    #         f.write(file_str_cbcr_ref[b])
    #     with open(f"golden/src_y_channel/src_ych_{int_exp(_)}_fy{b}.dat", "w") as f:
    #         f.write(file_str_y_src[b])
    #     with open(f"golden/src_cbcr_channel/src_cbcrch_{int_exp(_)}_fc{b}.dat", "w") as f:
    #         f.write(file_str_cbcr_src[b])
    #     with open(f"golden/light_match/light_match_{int_exp(_)}_fy{b}.dat", "w") as f:
    #         f.write(file_str_y_lumi[b])
    #     with open(f"golden/color_match/color_match_{int_exp(_)}_fc{b}.dat", "w") as f:
    #         f.write(file_str_cbcr_color[b])
        # with open(f"golden/final_{ref_idx}/final_{int_exp(_)}_fy{b}.dat", "w") as f:
        #     f.write(file_str_y_final[b])
        # with open(f"golden/final_{ref_idx}/final_{int_exp(_)}_fc{b}.dat", "w") as f:
        #     f.write(file_str_cbcr_final[b])

    return results


## Simple Demo (requires OpenCV)

Place `src.png` (source image) and `ref.png` (reference image) in the working directory.
Run the cell below (or run the notebook as a script) to generate `out_color_matched.png`.


In [18]:
import time
if __name__ == "__main__":
    try:
        import cv2

        ref = cv2.imread(f"TP/ref/a{ref_idx+1}.jpg")
        ref = cv2.cvtColor(ref, cv2.COLOR_BGR2RGB)
        if ref is None:
            print("Please put 'ref.png' in the working directory to run the demo.")
        else:

            for _ in range(0, 30):

                in_path = f"TP/src/pat_{int_exp(_)}.dng"
                out_path = f"TP/output/out_{int_exp(_)}.png"

                with rawpy.imread(in_path) as raw:
                    # 取得 Bayer RAW（線性域）
                    bayer   = raw.raw_image.astype(np.float32)
                    pattern = raw.raw_pattern          # [[0 1],[3 2]]
                    colors  = raw.color_desc.decode()  # 'RGBG'
                    bayer = bayer
                    H, W = bayer.shape

                    # ---- 1. 在 RAW 上算 gray-world 白平衡 gain（你現在這套）----
                    sum_c   = {'R': 0.0, 'G': 0.0, 'B': 0.0}
                    count_c = {'R': 0,   'G': 0,   'B': 0}

                    for dy in range(2):
                        for dx in range(2):
                            c_idx = pattern[dy, dx]      # 0/1/2/3
                            c     = colors[c_idx]        # 'R'/'G'/'B'
                            patch = np.uint16(bayer[dy::2, dx::2])
                            sum_c[c]   += patch.sum()
                            count_c[c] += patch.size

                    g_R = quan((sum_c["G"]/2) / sum_c["R"])
                    g_G = 0.890625
                    g_B = quan((sum_c["G"]/2) / sum_c["B"])

                    gains = {'R': g_R, 'G': g_G, 'B': g_B}

                    # ---- 2. 在 Bayer 上套白平衡（RGGB 固定寫法）----
                    bayer_wb = bayer.copy()

                    # RGGB:
                    # (0,0) R, (0,1) G
                    # (1,0) G, (1,1) B
                    bayer_wb[0::2, 0::2] *= gains['R']   # R
                    bayer_wb[0::2, 1::2] *= gains['G']   # G
                    bayer_wb[1::2, 0::2] *= gains['G']   # G
                    bayer_wb[1::2, 1::2] *= gains['B']   # B

                    # clip 回合理範圍，轉成 uint16 給 demosaic 用
                    bayer_wb = np.clip(bayer_wb, 0, 65535).astype(np.uint16)

                # ---- 3. 用 OpenCV 做 demosaic：RGGB → RGB（線性 RGB，尚未 gamma）----
                # 注意：OpenCV 預設是 BGR，我這裡直接用 RGB 版本

                rgb_lin = demosaic_rggb_bilinear(bayer_wb)  # float32 linear RGB

                # ---- 4. 自己在 numpy 裡做 sRGB gamma ----

                src = apply_better_gamma(rgb_lin, white_percent=96, gamma=2.2)

                # Write golden

                file_str_r = ["", "", "", ""]
                file_str_gb = ["", "", "", ""]
                file_str_wb_r = ["", "", "", ""]
                file_str_wb_gb = ["", "", "", ""]
                file_str_demosaic_r = ["", "", "", ""]
                file_str_demosaic_gb = ["", "", "", ""]
                file_str_gamma_r = ["", "", "", ""]
                file_str_gamma_gb = ["", "", "", ""]

                # for i in range(H):
                #     for j in range(W // 4):
                        # if (i % 2):
                        #     file_str_r[i % 4] += "".join(map(dec_to_hex, [0, 0, 0, 0])) + "\n"
                #             file_str_wb_r[i % 4] += "".join(map(dec_to_hex, [0, 0, 0, 0])) + "\n"
                        # else:
                        #     file_str_r[i % 4] += "".join(map(dec_to_hex, [bayer[i][j*4], 0, bayer[i][j*4 + 2], 0])) + "\n"
                #             file_str_wb_r[i % 4] += "".join(map(dec_to_hex, [bayer_wb[i][j*4]*4, 0, bayer_wb[i][j*4 + 2]*4, 0])) + "\n"
                        # if (i % 2):
                        #     file_str_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [bayer[i][j*4]*2**16, bayer[i][j*4 + 1], bayer[i][j*4 + 2]*2**16, bayer[i][j*4 + 3]])) + "\n"
                #             file_str_wb_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(bayer_wb[i][j*4])*2**16*4, bayer_wb[i][j*4 + 1]*4, int(bayer_wb[i][j*4 + 2])*2**16*4, bayer_wb[i][j*4 + 3]*4])) + "\n"
                        # else:
                        #     file_str_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [0, bayer[i][j*4 + 1]*2**16, 0, bayer[i][j*4 + 3]*2**16])) + "\n"
                #             file_str_wb_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [0, int(bayer_wb[i][j*4 + 1])*2**16*4, 0, int(bayer_wb[i][j*4 + 3])*2**16*4])) + "\n"
                #         file_str_demosaic_r[i % 4] += "".join(map(dec_to_hex, rgb_lin[:,:,0][i][j*4:j*4+4]*4)) + "\n"
                #         file_str_demosaic_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(rgb_lin[:,:,1][i][j*4]*4*2**16) + int(rgb_lin[:,:,2][i][j*4]*4), 
                #                                                                                 int(rgb_lin[:,:,1][i][j*4+1]*4*2**16) + int(rgb_lin[:,:,2][i][j*4+1]*4),
                #                                                                                 int(rgb_lin[:,:,1][i][j*4+2]*4*2**16) + int(rgb_lin[:,:,2][i][j*4+2]*4),
                #                                                                                 int(rgb_lin[:,:,1][i][j*4+3]*4*2**16) + int(rgb_lin[:,:,2][i][j*4+3]*4)])) + "\n"
                #         file_str_gamma_r[i % 4] += "".join(map(dec_to_hex, src[:,:,0][i][j*4:j*4+4])) + "\n"
                #         file_str_gamma_gb[i % 4] += "".join(map(lambda x: dec_to_hex(x, 32), [int(src[:,:,1][i][j*4])*2**16 + int(src[:,:,2][i][j*4]), 
                #                                                                                 int(src[:,:,1][i][j*4+1])*2**16 + int(src[:,:,2][i][j*4+1]),
                #                                                                                 int(src[:,:,1][i][j*4+2])*2**16 + int(src[:,:,2][i][j*4+2]),
                #                                                                                 int(src[:,:,1][i][j*4+3])*2**16 + int(src[:,:,2][i][j*4+3])])) + "\n"
                    # print(f"{_}th pat:" + str(round(i / H, 2)*100) + "%")

                # for b in range(4):
                #     with open(f"pat/src_raw_{int_exp(_)}_r{b}.dat", "w") as f:
                #         f.write(file_str_r[b])
                #     with open(f"pat/src_raw_{int_exp(_)}_gb{b}.dat", "w") as f:
                #         f.write(file_str_gb[b])
                #     with open(f"golden/wb/src_wb_{int_exp(_)}_r{b}.dat", "w") as f:
                #         f.write(file_str_wb_r[b])
                #     with open(f"golden/wb/src_wb_{int_exp(_)}_gb{b}.dat", "w") as f:
                #         f.write(file_str_wb_gb[b])
                #     with open(f"golden/demosaic/src_demosaic_{int_exp(_)}_r{b}.dat", "w") as f:
                #         f.write(file_str_demosaic_r[b])
                #     with open(f"golden/demosaic/src_demosaic_{int_exp(_)}_gb{b}.dat", "w") as f:
                #         f.write(file_str_demosaic_gb[b])
                #     with open(f"golden/gamma/src_gamma_{int_exp(_)}_r{b}.dat", "w") as f:
                #         f.write(file_str_gamma_r[b])
                #     with open(f"golden/gamma/src_gamma_{int_exp(_)}_gb{b}.dat", "w") as f:
                #         f.write(file_str_gamma_gb[b])

                res = color_match_pipeline_debug(src, ref, _=_)
                # cv2.imwrite("C:/Users/0607f/code/python/ICLAB/Final/result/step2_ycbcr.png",    res["ycbcr"])
                # cv2.imwrite("C:/Users/0607f/code/python/ICLAB/Final/result/step3_hist.png",     res["after_hist"])
                cv2.imwrite(out_path,    res["final"])
                print("save:", out_path)

    except ImportError:
        print("OpenCV is not installed. Install with `pip install opencv-python` to run the demo.")


save: TP/output/out_00.png
save: TP/output/out_01.png
save: TP/output/out_02.png
save: TP/output/out_03.png
save: TP/output/out_04.png
save: TP/output/out_05.png
save: TP/output/out_06.png
save: TP/output/out_07.png
save: TP/output/out_08.png


KeyboardInterrupt: 